## Test Error

In this notebook, we will compute the test MSE by performing linear regression on the features selected in "LinearRegressionWithAddedFeatures.ipynb".

### Importing the necessary Packages

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime

import statsmodels.api as sm

### Global variables

In [6]:
lat_min = 38
lat_max = 45
long_min = -90
long_max = -83

windowSize = 23
size = 5
fraction = 1/4
scale = 1.5
peak = 7

sigCols = ["IBA CODE",
           "PREVIOUS AVERAGE OBSERVATION",
           "ROLLING AVERAGE OBSERVATION",
           "INTERCEPT",
           "DURATION MINUTES_TRANSFORMED",
           "TIME OBSERVATIONS STARTED_TRANSFORMED"]

### Processing the data

In [8]:
trainData = pd.read_csv("Data/TrainData.csv")
testData = pd.read_csv("Data/TestData.csv")
trainStartDate = (np.datetime64(datetime.datetime(2019,1,1)) - np.datetime64(datetime.datetime(2018,1,1))).astype('timedelta64[D]') // np.timedelta64(1, 'D')
testStartDate = np.min(testData["OBSERVATION DATE"])

data = pd.concat([trainData, testData], axis = 0, ignore_index = True)

We will use the helper functions from "LinearRegressionWithAddedFeatures.ipynb"

In [10]:
def findWindowAverageByLoc(windowSize, data):
    cutoffDate = (np.datetime64(datetime.datetime(2019,1,1)) - np.datetime64(datetime.datetime(2018,1,1))).astype('timedelta64[D]') // np.timedelta64(1, 'D')
    
    temp = data.loc[:,["OBSERVATION DATE", "LATITUDE", "LONGITUDE", "OBSERVATION COUNT"]]
    temp = temp[temp["OBSERVATION DATE"] < cutoffDate]
    
    temp["LATITUDE ROUNDED"] = np.floor(temp["LATITUDE"])
    temp["LONGITUDE ROUNDED"] = np.floor(temp["LONGITUDE"])
    temp["NUM OBS"] = np.ones(len(temp))
    temp = temp.drop(["LATITUDE","LONGITUDE"], axis = 1)
    
    temp = temp.groupby(by = ["OBSERVATION DATE", "LATITUDE ROUNDED", "LONGITUDE ROUNDED"]).sum()
    res = temp.copy()
    
    for i in range(lat_max - lat_min):
        for j in range(long_max - long_min):
            cur = temp.loc[:,lat_min+i,long_min+j]        
            
            rAvg = cur.rolling(windowSize, min_periods = 1, center = True).sum()
            rAvg = rAvg.reset_index()
            rAvg["LATITUDE ROUNDED"] = lat_min+i
            rAvg["LONGITUDE ROUNDED"] = long_min+j
            rAvg = rAvg.set_index(["OBSERVATION DATE", "LATITUDE ROUNDED", "LONGITUDE ROUNDED"])
            
            res.loc[:,lat_min+i,long_min+j] = rAvg

    res = res.reset_index()
    res["PREVIOUS AVERAGE OBSERVATION"] = res["OBSERVATION COUNT"]/res["NUM OBS"]

    res = res.drop(["OBSERVATION COUNT", "NUM OBS"], axis = 1)
    
    return res


def addWindowAverageByLoc(windowAverage, data):
    windowAverage["OBSERVATION DATE"] = windowAverage["OBSERVATION DATE"] + 365

    data["LATITUDE ROUNDED"] = np.floor(data["LATITUDE"])
    data["LONGITUDE ROUNDED"] = np.floor(data["LONGITUDE"])
    
    res = data.merge(windowAverage, how = "left", on = ["OBSERVATION DATE", "LATITUDE ROUNDED", "LONGITUDE ROUNDED"])
    
    res = res.drop(["LATITUDE ROUNDED", "LONGITUDE ROUNDED"], axis = 1)

    cutoffDate = (np.datetime64(datetime.datetime(2019,1,1)) - np.datetime64(datetime.datetime(2018,1,1))).astype('timedelta64[D]') // np.timedelta64(1, 'D')
    fillValue = np.mean(data[data["OBSERVATION DATE"] < cutoffDate]["OBSERVATION COUNT"])
    
    res = res.fillna(fillValue)

    return res


def findRollingAverageByLoc(size, data):
    
    temp = data.loc[:,["OBSERVATION DATE", "LATITUDE", "LONGITUDE", "OBSERVATION COUNT"]]
    
    temp["LATITUDE ROUNDED"] = np.floor(temp["LATITUDE"])
    temp["LONGITUDE ROUNDED"] = np.floor(temp["LONGITUDE"])
    temp["NUM OBS"] = np.ones(len(temp))
    temp = temp.drop(["LATITUDE","LONGITUDE"], axis = 1)
    
    temp = temp.groupby(by = ["OBSERVATION DATE", "LATITUDE ROUNDED", "LONGITUDE ROUNDED"]).sum()
    res = temp.copy()
    
    for i in range(lat_max - lat_min):
        for j in range(long_max - long_min):
            cur = temp.loc[:,lat_min+i,long_min+j]        
            
            rAvg = cur.rolling(size, min_periods = 1, closed = "left").sum()
            rAvg = rAvg.reset_index()
            rAvg["LATITUDE ROUNDED"] = lat_min+i
            rAvg["LONGITUDE ROUNDED"] = long_min+j
            rAvg = rAvg.set_index(["OBSERVATION DATE", "LATITUDE ROUNDED", "LONGITUDE ROUNDED"])
            
            res.loc[:,lat_min+i,long_min+j] = rAvg

    res = res.reset_index()
    res["ROLLING AVERAGE OBSERVATION"] = res["OBSERVATION COUNT"]/res["NUM OBS"]

    res = res.drop(["OBSERVATION COUNT", "NUM OBS"], axis = 1)
    
    return res




def addRollingAverageByLoc(rollingAverage, data):

    data["LATITUDE ROUNDED"] = np.floor(data["LATITUDE"])
    data["LONGITUDE ROUNDED"] = np.floor(data["LONGITUDE"])
    
    res = data.merge(rollingAverage, how = "left", on = ["OBSERVATION DATE", "LATITUDE ROUNDED", "LONGITUDE ROUNDED"])
    
    res = res.drop(["LATITUDE ROUNDED", "LONGITUDE ROUNDED"], axis = 1)
    return res



def getExponentialModel(column, scale, peak):

    ## transform the given column using the piecewise defined function f(t)
    
    ratio = (24 - peak)/peak

    res = pd.Series(index = column.index)

    beforeIndex = column[column <= peak].index
    res[beforeIndex] = np.exp(-(peak - column.loc[beforeIndex])/scale)
    
    afterIndex = column[column > peak].index
    res[afterIndex] = np.exp(-(column.loc[afterIndex] - peak)/(ratio * scale))

    return res

In [11]:
averaged = findWindowAverageByLoc(windowSize, data)
newData = addWindowAverageByLoc(averaged, data)

rolling = findRollingAverageByLoc(size,data)
newData = addRollingAverageByLoc(rolling, newData)

newData = newData[newData["OBSERVATION DATE"] >= trainStartDate]
newData["INTERCEPT"] = np.ones(len(newData))
newData["DURATION MINUTES_TRANSFORMED"] = np.power(newData["DURATION MINUTES"].to_numpy(), fraction)
newData["TIME OBSERVATIONS STARTED_TRANSFORMED"] = getExponentialModel(newData["TIME OBSERVATIONS STARTED"], scale, peak)
newData["IBA CODE"] = newData["IBA CODE"].astype(int)

trainData = newData[newData["OBSERVATION DATE"] < testStartDate]
testData = newData[newData["OBSERVATION DATE"] >= testStartDate]

trainX = trainData[sigCols]
trainY = trainData["OBSERVATION COUNT"]

testX = testData[sigCols]
testY = testData["OBSERVATION COUNT"]

### Calculating the test MSE

In [13]:
model = sm.OLS(trainY, trainX)
results = model.fit()
preds = results.predict(testX)
testMSE = np.mean(np.power(preds - testY.values,2))

print(f"The test MSE is: {testMSE}")

The test MSE is: 22.034577693630567


In [14]:
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:      OBSERVATION COUNT   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.049
Method:                 Least Squares   F-statistic:                     3938.
Date:                Sat, 22 Aug 2026   Prob (F-statistic):               0.00
Time:                        17:28:44   Log-Likelihood:            -1.2418e+06
No. Observations:              378210   AIC:                         2.484e+06
Df Residuals:                  378204   BIC:                         2.484e+06
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------